### Notebook for developing Poisson kernel-based distribution

In [ ]:
import numpy as np
import spheremix as smix
import matplotlib.pyplot as plt

import spheroids

from loguru import logger

#### Generate samples on the unit sphere

In [ ]:
# ---------------------------
# Example: 100 samples in 4000D
# ---------------------------
rng = np.random.default_rng(42)

N = 10
D = 512
rho = 0.10

# Create a random unit vector mu
mu = rng.normal(size=D)
mu /= np.linalg.norm(mu)

# Sample data
X = smix.distributions.sample_spherical_cauchy(N, mu, rho, rng=rng)

# Evaluate densities
logpdf_vals: np.ndarray = smix.distributions.pkbd_logpdf(X, mu, rho)

# TODO: Compare log-PDF values from spheremix and spheroids
loglik_spheremix: np.ndarray = smix.distributions.pkbd_log_kernel(X, mu, rho)
loglik_spheroids = spheroids.PKBD.log_likelihood(X, mu, rho)

logger.info(f"Spheremix PKBD (unnormalized) log-densities: {loglik_spheremix}")
logger.info(f"Spheroids PKBD (unnormalized) log-densities: {loglik_spheroids}")


print("X shape:", X.shape)
print("Norms close to 1:", np.allclose(np.linalg.norm(X, axis=1), 1.0, atol=1e-8))
print("logpdf range:", logpdf_vals.min(), logpdf_vals.max())

# Useful diagnostic: alignment with mu
cos_vals = X @ mu

# ---------------------------
# Visualization
# ---------------------------
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))

axes[0].hist(cos_vals, bins=20, color="steelblue", edgecolor="black")
axes[0].set_title("Cosine similarity to mu")
axes[0].set_xlabel("x · mu")
axes[0].set_ylabel("Count")

axes[1].hist(logpdf_vals, bins=20, color="darkorange", edgecolor="black")
axes[1].set_title("Log-PDF values")
axes[1].set_xlabel("log f(x)")
axes[1].set_ylabel("Count")

axes[2].scatter(cos_vals, logpdf_vals, s=30, alpha=0.8, color="crimson")
axes[2].set_title("Log-PDF vs x · mu")
axes[2].set_xlabel("x · mu")
axes[2].set_ylabel("log f(x)")

plt.tight_layout()
plt.show()